# Comparative Study of Deep Learning Architectures for Khmer ASR
### Controlled Google Colab rerun workflow

**Course:** Deep Learning Final Project (2026–2027)  
**Student:** Khemrak Pasey  
**Dataset:** Google FLEURS Khmer (`km_kh`)

The saved 83.89%, 83.67%, and 15.96% CER results came from earlier runs with different training selections. They remain diagnostic. This notebook creates one fixed row manifest, then trains Whisper-Tiny and MMS-1B on those same train, validation, and test examples. Only report new matched scores after both runs and saved evidence finish.


## 1. Hardware & GPU Check
Make sure your Colab session has a GPU assigned: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Running on CPU. Please switch to T4 GPU in Runtime settings.')

## 2. Install the project dependencies
Keep cached datasets and model checkpoints while training.


In [ ]:
# Keep Colab caches and checkpoints while the runs are in progress.
!pip install -q torch torchaudio transformers datasets torchcodec evaluate jiwer accelerate tensorboard soundfile librosa matplotlib python-pptx av


## 3. Open the current GitHub project
The repository contains the shared-split workflow. Run this cell in a fresh Colab session before training.


In [ ]:
from pathlib import Path
%cd /content
if not Path('/content/khmer_asr/.git').exists():
    !git clone https://github.com/Seypa-47/khmer_asr.git
%cd /content/khmer_asr
!git pull --ff-only


## 3A. Mount Google Drive for persistent final checkpoints
Colab local files disappear when its session ends. Approve the Drive prompt, then train final checkpoints into the dedicated folder below. Pilot tuning histories remain small and are copied later.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PERSISTENT_ROOT = Path('/content/drive/MyDrive/khmer_asr_final_runs')
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
WHISPER_DIR = str(PERSISTENT_ROOT / 'whisper-tiny-khmer-matched')
MMS_DIR = str(PERSISTENT_ROOT / 'mms-khmer-ctc-matched')
print('Final checkpoints will be saved in:', PERSISTENT_ROOT)


## 4. Create the shared FLEURS row manifest
Run this once before training either model. Both scripts load exactly these row indices. Test candidates start at row 200 because historical experiments already inspected rows 0–199.


In [ ]:
!python src/matched_fleurs.py --output results/matched_fleurs_split.json --seed 42 --train-candidates 1000 --validation-candidates 200 --test-start 200 --test-candidates 200 --max-duration-seconds 10


## 5. Run Approach 1: Whisper-Tiny full fine-tuning
Use the saved row manifest. Save to a new folder so older checkpoints remain intact. The training state and metrics will be copied into `results/`.


In [ ]:
!python src/finetune_whisper.py \
  --output-dir {WHISPER_DIR} \
  --model-name openai/whisper-tiny --use-fleurs-train --skip-ddd \
  --split-manifest results/matched_fleurs_split.json \
  --metrics-output results/whisper_matched_metrics.json \
  --trainer-state-output results/whisper_matched_trainer_state.json \
  --seed 42 --num-train-epochs 3 --learning-rate 1e-5 --weight-decay 0.0 \
  --warmup-steps 35 --per-device-train-batch-size 1 \
  --per-device-eval-batch-size 4 --gradient-accumulation-steps 8 \
  --logging-steps 25 --eval-steps 118 --save-steps 118 --fp16


## 6. Tune MMS on validation data only
These short pilot runs use the same training and validation rows. They save metrics and histories but no large pilot weights. Compare two learning rates and two weight-decay settings. Do not inspect the test set to choose settings.


In [ ]:
import json, subprocess
from pathlib import Path
trials = [
    ('baseline', '5e-5', '0.0'),
    ('lower_lr', '3e-5', '0.0'),
    ('weight_decay', '5e-5', '0.01'),
]
tuning = []
for name, lr, wd in trials:
    state_path = f'results/mms_pilot_{name}_trainer_state.json'
    command = [
        'python', 'src/train_mms.py', '--output-dir', f'models/mms-pilot-{name}',
        '--model-id', 'facebook/mms-1b-all', '--target-lang', 'khm',
        '--split-manifest', 'results/matched_fleurs_split.json',
        '--metrics-output', f'results/mms_pilot_{name}_metrics.json',
        '--trainer-state-output', state_path,
        '--seed', '42', '--num-train-epochs', '2', '--learning-rate', lr,
        '--weight-decay', wd, '--unfreeze-top-layers', '4', '--apply-spec-augment',
        '--lr-scheduler-type', 'cosine', '--warmup-steps', '50',
        '--per-device-train-batch-size', '1', '--per-device-eval-batch-size', '1',
        '--gradient-accumulation-steps', '8', '--eval-steps', '50',
        '--logging-steps', '10', '--fp16', '--tuning-only', '--skip-test',
    ]
    subprocess.run(command, check=True)
    history = json.loads(Path(state_path).read_text(encoding='utf-8'))['log_history']
    scores = [float(row['eval_cer']) for row in history if 'eval_cer' in row]
    if not scores:
        raise RuntimeError(f'No validation CER was recorded for {name}')
    tuning.append({'name': name, 'learning_rate': lr, 'weight_decay': wd,
                   'best_validation_cer': min(scores)})
    print(tuning[-1])
Path('results/mms_tuning_summary.json').write_text(json.dumps(tuning, indent=2), encoding='utf-8')
chosen = min(tuning, key=lambda row: row['best_validation_cer'])
BEST_LR = chosen['learning_rate']
BEST_WD = chosen['weight_decay']
print('Use for the full MMS run:', chosen)


## 7. Run the full MMS experiment with the chosen settings
This command uses `BEST_LR` and `BEST_WD` from the validation-only pilot cell. It saves a separate final checkpoint, history, and test metrics.


In [ ]:
!python src/train_mms.py \
  --output-dir {MMS_DIR} \
  --model-id facebook/mms-1b-all --target-lang khm \
  --split-manifest results/matched_fleurs_split.json \
  --metrics-output results/mms_matched_metrics.json \
  --trainer-state-output results/mms_matched_trainer_state.json \
  --seed 42 --num-train-epochs 15 --learning-rate {BEST_LR} --weight-decay {BEST_WD} \
  --unfreeze-top-layers 4 --apply-spec-augment --lr-scheduler-type cosine \
  --warmup-steps 50 --per-device-train-batch-size 1 \
  --per-device-eval-batch-size 1 --gradient-accumulation-steps 8 \
  --eval-steps 50 --save-steps 50 --logging-steps 10 --fp16


## Optional third approach: frozen encoder
There is no saved checkpoint or score for this ablation. Do not report it as a trained approach unless a separate run completes and its evidence is saved.


## 8. Audit the matched outputs and make figures
Run this only after the full Whisper and MMS training cells finish. The plots use new matched-run files and never substitute older diagnostic results.


In [ ]:
!python src/evaluate_saved_whisper.py \
  --model-dir {WHISPER_DIR} \
  --split-manifest results/matched_fleurs_split.json \
  --output results/whisper_matched_predictions.json --device cuda
!python src/evaluate_saved_mms.py \
  --model-dir {MMS_DIR} \
  --split-manifest results/matched_fleurs_split.json \
  --output results/mms_matched_predictions.json --device cuda
!python src/plot_matched_results.py

from IPython.display import Image, display
display(Image('results/matched_learning_curves.png'))
display(Image('results/matched_metrics_comparison.png'))


## 9. Preserve evidence and share the weights
Final model folders are already on Google Drive. Copy the small results folder there too, then share both model folders with the lecturer and add verified download links to README.md. Keep the pilot metrics and histories as evidence. Do not delete checkpoints until the final results and figures are reviewed.


In [ ]:
import shutil
from pathlib import Path
wanted = (
    'matched_fleurs_split.json', 'mms_tuning_summary.json',
    'whisper_matched_metrics.json', 'mms_matched_metrics.json',
    'whisper_matched_trainer_state.json', 'mms_matched_trainer_state.json',
    'whisper_matched_predictions.json', 'mms_matched_predictions.json',
    'matched_learning_curves.png', 'matched_metrics_comparison.png',
)
evidence_dir = PERSISTENT_ROOT / 'results'
evidence_dir.mkdir(parents=True, exist_ok=True)
for name in wanted:
    source = Path('results') / name
    if not source.is_file():
        raise FileNotFoundError(f'Missing evidence: {source}')
    shutil.copy2(source, evidence_dir / name)
for name in ('baseline', 'lower_lr', 'weight_decay'):
    for suffix in ('metrics.json', 'trainer_state.json'):
        source = Path('results') / f'mms_pilot_{name}_{suffix}'
        shutil.copy2(source, evidence_dir / source.name)
print('Saved evidence:', evidence_dir)
print('Whisper checkpoint:', Path(WHISPER_DIR, 'model.safetensors').is_file())
print('MMS checkpoint:', Path(MMS_DIR, 'model.safetensors').is_file())
